In [2]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from time import perf_counter
from scipy.sparse import load_npz

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

In [3]:
RUTA_PROYECTO = Path("..")

RUTA_VECTORIZACION = (
    RUTA_PROYECTO
    / "models"
    / "primera_iteracion"
    / "vectorizacion"
)

RUTA_RANDOM_FOREST = (
    RUTA_PROYECTO
    / "models"
    / "segunda_iteracion"
    / "random_forest"
)

RUTA_RANDOM_FOREST.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Carpeta de Random Forest:",
    RUTA_RANDOM_FOREST.resolve()
)

Carpeta de Random Forest: C:\Users\Paula María\Desktop\IS\5º CURSO\TFG\Trabajo-Fin-de-Grado\models\segunda_iteracion\random_forest


In [4]:
X_train_tfidf = load_npz(
    RUTA_VECTORIZACION
    / "X_train_tfidf.npz"
)

X_test_tfidf = load_npz(
    RUTA_VECTORIZACION
    / "X_test_tfidf.npz"
)

y_train = np.load(
    RUTA_VECTORIZACION
    / "y_train.npy",
    allow_pickle=True
)

y_test = np.load(
    RUTA_VECTORIZACION
    / "y_test.npy",
    allow_pickle=True
)

datos_test = pd.read_parquet(
    RUTA_VECTORIZACION
    / "datos_test.parquet"
)

In [5]:
print("COMPROBACIÓN DE LOS DATOS")
print("-------------------------")

print(
    "Matriz de entrenamiento:",
    X_train_tfidf.shape
)

print(
    "Matriz de prueba:",
    X_test_tfidf.shape
)

print(
    "¿Coincide entrenamiento?",
    X_train_tfidf.shape[0] == len(y_train)
)

print(
    "¿Coincide prueba?",
    X_test_tfidf.shape[0] == len(y_test)
)

print(
    "¿Mismo número de características?",
    X_train_tfidf.shape[1]
    == X_test_tfidf.shape[1]
)

COMPROBACIÓN DE LOS DATOS
-------------------------
Matriz de entrenamiento: (1188274, 50000)
Matriz de prueba: (297069, 50000)
¿Coincide entrenamiento? True
¿Coincide prueba? True
¿Mismo número de características? True


In [9]:
# Configuración inicial controlada
modelo_random_forest = RandomForestClassifier(
    n_estimators=100, # construye 100 árboles.
    max_depth=30, # limita la profundidad para controlar coste y sobreajuste
    min_samples_split=10, # exige al menos diez registros para dividir un nodo
    min_samples_leaf=5, # exige al menos cinco registros en cada hoja
    max_features="sqrt", # cada división examina la raíz cuadrada del número de características
    class_weight=None, 
    max_samples=0.40, # cada árbol se entrena con una submuestra del 40 %
    n_jobs=-1,
    random_state=42, # permite reproducir el entrenamiento
    verbose=1
)

In [10]:
# Entrenamiento inicial

inicio_entrenamiento = perf_counter()

modelo_random_forest.fit(
    X_train_tfidf,
    y_train
)

tiempo_entrenamiento_rf = (
    perf_counter() - inicio_entrenamiento
)

print("\nENTRENAMIENTO RANDOM FOREST COMPLETADO")
print("--------------------------------------")

print(
    "Tiempo:",
    f"{tiempo_entrenamiento_rf:.2f} segundos"
)

print(
    "Tiempo en minutos:",
    f"{tiempo_entrenamiento_rf / 60:.2f}"
)

print(
    "Árboles entrenados:",
    len(modelo_random_forest.estimators_)
)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   54.1s



ENTRENAMIENTO RANDOM FOREST COMPLETADO
--------------------------------------
Tiempo: 132.85 segundos
Tiempo en minutos: 2.21
Árboles entrenados: 100


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  2.1min finished


In [11]:
# Generar predicciones

inicio_prediccion = perf_counter()

predicciones_rf = (
    modelo_random_forest.predict(
        X_test_tfidf
    )
)

tiempo_prediccion_rf = (
    perf_counter() - inicio_prediccion
)

print("\nPREDICCIÓN COMPLETADA")
print("---------------------")

print(
    "Reseñas clasificadas:",
    f"{len(predicciones_rf):,}"
)

print(
    "Tiempo:",
    f"{tiempo_prediccion_rf:.2f} segundos"
)

print(
    "¿Hay una predicción por reseña?",
    len(predicciones_rf) == len(y_test)
)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.7s



PREDICCIÓN COMPLETADA
---------------------
Reseñas clasificadas: 297,069
Tiempo: 2.06 segundos
¿Hay una predicción por reseña? True


[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.9s finished


In [12]:
# Métricas
accuracy_rf = accuracy_score(
    y_test,
    predicciones_rf
)

precision_macro_rf = precision_score(
    y_test,
    predicciones_rf,
    average="macro",
    zero_division=0
)

recall_macro_rf = recall_score(
    y_test,
    predicciones_rf,
    average="macro",
    zero_division=0
)

f1_macro_rf = f1_score(
    y_test,
    predicciones_rf,
    average="macro",
    zero_division=0
)

print("\nMÉTRICAS DE RANDOM FOREST")
print("-------------------------")

print(
    f"Exactitud:           {accuracy_rf:.4f}"
)

print(
    f"Precisión macro:     {precision_macro_rf:.4f}"
)

print(
    f"Exhaustividad macro: {recall_macro_rf:.4f}"
)

print(
    f"F1 macro:            {f1_macro_rf:.4f}"
)


MÉTRICAS DE RANDOM FOREST
-------------------------
Exactitud:           0.7735
Precisión macro:     0.5789
Exhaustividad macro: 0.3615
F1 macro:            0.3420


In [14]:
informe_rf = classification_report(
    y_test,
    predicciones_rf,
    labels=[
        "negativo",
        "neutral",
        "positivo"
    ],
    digits=4,
    output_dict=True,
    zero_division=0
)

informe_rf_df = (
    pd.DataFrame(informe_rf)
    .transpose()
)

informe_rf_df.round(4)

,precision,recall,f1-score,support
negativo,0.9660,0.0846,0.1556,47697.0000
neutral,0.0000,0.0000,0.0000,23558.0000
positivo,0.7708,0.9998,0.8705,225814.0000
accuracy,0.7735,0.7735,0.7735,0.7735
macro avg,0.5789,0.3615,0.3420,297069.0000
weighted avg,0.7410,0.7735,0.6867,297069.0000


In [15]:
# Optimización de Random Forest

configuraciones_rf = [
    {
        "nombre": "configuracion_inicial",
        "max_depth": 30,
        "class_weight": None
    },
    {
        "nombre": "clases_balanceadas",
        "max_depth": 30,
        "class_weight": "balanced"
    },
    {
        "nombre": "balanceado_mayor_profundidad",
        "max_depth": 50,
        "class_weight": "balanced"
    }
]

In [17]:
# División entrenamiento interno

from sklearn.model_selection import train_test_split

(
    X_train_interno_rf,
    X_validacion_rf,
    y_train_interno_rf,
    y_validacion_rf
) = train_test_split(
    X_train_tfidf,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

In [18]:
resultados_ajuste_rf = []

for numero, configuracion in enumerate(
    configuraciones_rf,
    start=1
):
    print(
        f"\nCONFIGURACIÓN {numero}/"
        f"{len(configuraciones_rf)}"
    )
    print(configuracion["nombre"])
    print("-" * 50)

    modelo_temporal = RandomForestClassifier(
        n_estimators=50,
        max_depth=configuracion["max_depth"],
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight=configuracion["class_weight"],
        max_samples=0.40,
        n_jobs=-1,
        random_state=42,
        verbose=0
    )

    inicio_entrenamiento = perf_counter()

    modelo_temporal.fit(
        X_train_interno_rf,
        y_train_interno_rf
    )

    tiempo_entrenamiento = (
        perf_counter() - inicio_entrenamiento
    )

    inicio_prediccion = perf_counter()

    predicciones_validacion = modelo_temporal.predict(
        X_validacion_rf
    )

    tiempo_prediccion = (
        perf_counter() - inicio_prediccion
    )

    accuracy = accuracy_score(
        y_validacion_rf,
        predicciones_validacion
    )

    precision_macro = precision_score(
        y_validacion_rf,
        predicciones_validacion,
        average="macro",
        zero_division=0
    )

    recall_macro = recall_score(
        y_validacion_rf,
        predicciones_validacion,
        average="macro",
        zero_division=0
    )

    f1_macro = f1_score(
        y_validacion_rf,
        predicciones_validacion,
        average="macro",
        zero_division=0
    )

    informe = classification_report(
        y_validacion_rf,
        predicciones_validacion,
        labels=[
            "negativo",
            "neutral",
            "positivo"
        ],
        output_dict=True,
        zero_division=0
    )

    resultados_ajuste_rf.append({
        "Configuración": configuracion["nombre"],
        "max_depth": configuracion["max_depth"],
        "class_weight": str(
            configuracion["class_weight"]
        ),
        "Accuracy": accuracy,
        "Precision macro": precision_macro,
        "Recall macro": recall_macro,
        "F1 macro": f1_macro,
        "F1 negativo": informe[
            "negativo"
        ]["f1-score"],
        "F1 neutral": informe[
            "neutral"
        ]["f1-score"],
        "F1 positivo": informe[
            "positivo"
        ]["f1-score"],
        "Tiempo entrenamiento": tiempo_entrenamiento,
        "Tiempo predicción": tiempo_prediccion
    })

    print(f"Exactitud:           {accuracy:.4f}")
    print(f"Precisión macro:     {precision_macro:.4f}")
    print(f"Exhaustividad macro: {recall_macro:.4f}")
    print(f"F1 macro:            {f1_macro:.4f}")
    print(
        "F1 negativo:",
        f"{informe['negativo']['f1-score']:.4f}"
    )
    print(
        "F1 neutral:",
        f"{informe['neutral']['f1-score']:.4f}"
    )
    print(
        "F1 positivo:",
        f"{informe['positivo']['f1-score']:.4f}"
    )
    print(
        "Tiempo:",
        f"{tiempo_entrenamiento}"
    )


CONFIGURACIÓN 1/3
configuracion_inicial
--------------------------------------------------
Exactitud:           0.7762
Precisión macro:     0.5755
Exhaustividad macro: 0.3671
F1 macro:            0.3519
F1 negativo: 0.1840
F1 neutral: 0.0000
F1 positivo: 0.8718
Tiempo: 41.39862319995882

CONFIGURACIÓN 2/3
clases_balanceadas
--------------------------------------------------
Exactitud:           0.7351
Precisión macro:     0.5869
Exhaustividad macro: 0.6435
F1 macro:            0.5932
F1 negativo: 0.6562
F1 neutral: 0.2814
F1 positivo: 0.8420
Tiempo: 54.95162159996107

CONFIGURACIÓN 3/3
balanceado_mayor_profundidad
--------------------------------------------------
Exactitud:           0.7458
Precisión macro:     0.5991
Exhaustividad macro: 0.6683
F1 macro:            0.6095
F1 negativo: 0.6816
F1 neutral: 0.2977
F1 positivo: 0.8494
Tiempo: 116.48706040001707


In [19]:
# Modelo final

modelo_rf_optimizado = RandomForestClassifier(
    n_estimators=100,
    max_depth=50,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced",
    max_samples=0.40,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

inicio_entrenamiento = perf_counter()

modelo_rf_optimizado.fit(
    X_train_tfidf,
    y_train
)

tiempo_entrenamiento_rf_optimizado = (
    perf_counter() - inicio_entrenamiento
)

print("\nENTRENAMIENTO COMPLETADO")
print("------------------------")
print(
    "Tiempo:",
    f"{tiempo_entrenamiento_rf_optimizado:.2f} segundos"
)
print(
    "Tiempo en minutos:",
    f"{tiempo_entrenamiento_rf_optimizado / 60:.2f}"
)
print(
    "Árboles entrenados:",
    len(modelo_rf_optimizado.estimators_)
)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  1.6min



ENTRENAMIENTO COMPLETADO
------------------------
Tiempo: 337.67 segundos
Tiempo en minutos: 5.63
Árboles entrenados: 100


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  5.5min finished


In [20]:
# Predicciones

inicio_prediccion = perf_counter()

predicciones_rf_optimizado = (
    modelo_rf_optimizado.predict(
        X_test_tfidf
    )
)

tiempo_prediccion_rf_optimizado = (
    perf_counter() - inicio_prediccion
)

print("\nPREDICCIÓN COMPLETADA")
print("---------------------")
print(
    "Reseñas clasificadas:",
    f"{len(predicciones_rf_optimizado):,}"
)
print(
    "Tiempo:",
    f"{tiempo_prediccion_rf_optimizado:.2f} segundos"
)
print(
    "¿Hay una predicción por reseña?",
    len(predicciones_rf_optimizado) == len(y_test)
)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.7s



PREDICCIÓN COMPLETADA
---------------------
Reseñas clasificadas: 297,069
Tiempo: 2.12 segundos
¿Hay una predicción por reseña? True


[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.0s finished


In [21]:
# Métricas
inicio_prediccion = perf_counter()

predicciones_rf_optimizado = (
    modelo_rf_optimizado.predict(
        X_test_tfidf
    )
)

tiempo_prediccion_rf_optimizado = (
    perf_counter() - inicio_prediccion
)

print("\nPREDICCIÓN COMPLETADA")
print("---------------------")
print(
    "Reseñas clasificadas:",
    f"{len(predicciones_rf_optimizado):,}"
)
print(
    "Tiempo:",
    f"{tiempo_prediccion_rf_optimizado:.2f} segundos"
)
print(
    "¿Hay una predicción por reseña?",
    len(predicciones_rf_optimizado) == len(y_test)
)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.8s



PREDICCIÓN COMPLETADA
---------------------
Reseñas clasificadas: 297,069
Tiempo: 2.56 segundos
¿Hay una predicción por reseña? True


[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.4s finished


In [22]:
# Informe por clase

inicio_prediccion = perf_counter()

predicciones_rf_optimizado = (
    modelo_rf_optimizado.predict(
        X_test_tfidf
    )
)

tiempo_prediccion_rf_optimizado = (
    perf_counter() - inicio_prediccion
)

print("\nPREDICCIÓN COMPLETADA")
print("---------------------")
print(
    "Reseñas clasificadas:",
    f"{len(predicciones_rf_optimizado):,}"
)
print(
    "Tiempo:",
    f"{tiempo_prediccion_rf_optimizado:.2f} segundos"
)
print(
    "¿Hay una predicción por reseña?",
    len(predicciones_rf_optimizado) == len(y_test)
)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.8s



PREDICCIÓN COMPLETADA
---------------------
Reseñas clasificadas: 297,069
Tiempo: 2.60 segundos
¿Hay una predicción por reseña? True


[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.5s finished


In [23]:
# Métricas
accuracy_rf_optimizado = accuracy_score(
    y_test,
    predicciones_rf_optimizado
)

precision_macro_rf_optimizado = precision_score(
    y_test,
    predicciones_rf_optimizado,
    average="macro",
    zero_division=0
)

recall_macro_rf_optimizado = recall_score(
    y_test,
    predicciones_rf_optimizado,
    average="macro",
    zero_division=0
)

f1_macro_rf_optimizado = f1_score(
    y_test,
    predicciones_rf_optimizado,
    average="macro",
    zero_division=0
)

print("\nMÉTRICAS DE RANDOM FOREST OPTIMIZADO")
print("------------------------------------")
print(f"Exactitud:           {accuracy_rf_optimizado:.4f}")
print(f"Precisión macro:     {precision_macro_rf_optimizado:.4f}")
print(f"Exhaustividad macro: {recall_macro_rf_optimizado:.4f}")
print(f"F1 macro:            {f1_macro_rf_optimizado:.4f}")


MÉTRICAS DE RANDOM FOREST OPTIMIZADO
------------------------------------
Exactitud:           0.7488
Precisión macro:     0.6064
Exhaustividad macro: 0.6819
F1 macro:            0.6176


In [24]:
# Por clase

informe_rf_optimizado = classification_report(
    y_test,
    predicciones_rf_optimizado,
    labels=[
        "negativo",
        "neutral",
        "positivo"
    ],
    digits=4,
    output_dict=True,
    zero_division=0
)

informe_rf_optimizado_df = (
    pd.DataFrame(informe_rf_optimizado)
    .transpose()
)

informe_rf_optimizado_df.round(4)

,precision,recall,f1-score,support
negativo,0.6596,0.7303,0.6932,47697.0000
neutral,0.2164,0.5410,0.3091,23558.0000
positivo,0.9433,0.7743,0.8505,225814.0000
accuracy,0.7488,0.7488,0.7488,0.7488
macro avg,0.6064,0.6819,0.6176,297069.0000
weighted avg,0.8401,0.7488,0.7823,297069.0000


In [25]:
# Guardar

RUTA_RF_OPTIMIZADO = (
    RUTA_RANDOM_FOREST
    / "modelo_optimizado"
)

RUTA_RF_OPTIMIZADO.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Carpeta de Random Forest optimizado:",
    RUTA_RF_OPTIMIZADO.resolve()
)

joblib.dump(
    modelo_rf_optimizado,
    RUTA_RF_OPTIMIZADO
    / "modelo_random_forest_optimizado.joblib"
)


Carpeta de Random Forest optimizado: C:\Users\Paula María\Desktop\IS\5º CURSO\TFG\Trabajo-Fin-de-Grado\models\segunda_iteracion\random_forest\modelo_optimizado


['..\\models\\segunda_iteracion\\random_forest\\modelo_optimizado\\modelo_random_forest_optimizado.joblib']

In [26]:
# Guardar  predicciones
np.save(
    RUTA_RF_OPTIMIZADO
    / "predicciones_random_forest_optimizado.npy",
    predicciones_rf_optimizado
)

In [27]:
# Métricas y configuración
metricas_rf_optimizado = {
    "modelo": "Random Forest optimizado",
    "n_estimators": int(
        modelo_rf_optimizado.n_estimators
    ),
    "max_depth": int(
        modelo_rf_optimizado.max_depth
    ),
    "min_samples_split": int(
        modelo_rf_optimizado.min_samples_split
    ),
    "min_samples_leaf": int(
        modelo_rf_optimizado.min_samples_leaf
    ),
    "max_features": (
        modelo_rf_optimizado.max_features
    ),
    "class_weight": (
        modelo_rf_optimizado.class_weight
    ),
    "max_samples": float(
        modelo_rf_optimizado.max_samples
    ),
    "numero_arboles_entrenados": int(
        len(modelo_rf_optimizado.estimators_)
    ),
    "accuracy": float(
        accuracy_rf_optimizado
    ),
    "precision_macro": float(
        precision_macro_rf_optimizado
    ),
    "recall_macro": float(
        recall_macro_rf_optimizado
    ),
    "f1_macro": float(
        f1_macro_rf_optimizado
    ),
    "tiempo_entrenamiento_segundos": float(
        tiempo_entrenamiento_rf_optimizado
    ),
    "tiempo_prediccion_segundos": float(
        tiempo_prediccion_rf_optimizado
    ),
    "numero_resenas_entrenamiento": int(
        len(y_train)
    ),
    "numero_resenas_prueba": int(
        len(y_test)
    ),
    "numero_caracteristicas": int(
        X_train_tfidf.shape[1]
    )
}

with open(
    RUTA_RF_OPTIMIZADO
    / "metricas_random_forest_optimizado.json",
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        metricas_rf_optimizado,
        archivo,
        indent=4,
        ensure_ascii=False
    )


In [28]:
# informe clasificacion por clase
informe_rf_optimizado = classification_report(
    y_test,
    predicciones_rf_optimizado,
    labels=[
        "negativo",
        "neutral",
        "positivo"
    ],
    output_dict=True,
    zero_division=0
)

informe_rf_optimizado_df = (
    pd.DataFrame(
        informe_rf_optimizado
    )
    .transpose()
)

informe_rf_optimizado_df.to_csv(
    RUTA_RF_OPTIMIZADO
    / "informe_clasificacion_random_forest_optimizado.csv"
)


In [ ]:
# Matriz de confusión

orden_clases = [
    "negativo",
    "neutral",
    "positivo"
]

figura, eje = plt.subplots(
    figsize=(8, 6)
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predicciones_rf_optimizado,
    labels=orden_clases,
    cmap="Blues",
    values_format=",",
    ax=eje
)

eje.set_title(
    "Matriz de confusión de Random Forest optimizado"
)

figura.tight_layout()

figura.savefig(
    RUTA_RF_OPTIMIZADO
    / "matriz_confusion_random_forest_optimizado.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(figura)


In [31]:
# Matriz de confusión normalizada

figura_normalizada, eje_normalizado = plt.subplots(
    figsize=(8, 6)
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predicciones_rf_optimizado,
    labels=orden_clases,
    normalize="true",
    cmap="Blues",
    values_format=".2%",
    ax=eje_normalizado
)

eje_normalizado.set_title(
    "Matriz de confusión normalizada "
    "de Random Forest optimizado"
)

figura_normalizada.tight_layout()

figura_normalizada.savefig(
    RUTA_RF_OPTIMIZADO
    / "matriz_confusion_normalizada_random_forest_optimizado.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(figura_normalizada)


In [32]:
# Predicciones con los textos

datos_test_rf = datos_test.copy()

datos_test_rf[
    "sentiment_predicted"
] = predicciones_rf_optimizado

datos_test_rf[
    "prediccion_correcta"
] = (
    datos_test_rf["sentiment"]
    == datos_test_rf["sentiment_predicted"]
)

datos_test_rf.to_parquet(
    RUTA_RF_OPTIMIZADO
    / "resultados_predicciones_random_forest_optimizado.parquet",
    index=False
)

In [33]:

# Comparación de configuraciones

resultados_ajuste_rf_df = (
    pd.DataFrame(resultados_ajuste_rf)
    .sort_values(
        by="F1 macro",
        ascending=False
    )
    .reset_index(drop=True)
)

resultados_ajuste_rf_df.round(4)

resultados_ajuste_rf_df.to_csv(
    RUTA_RANDOM_FOREST
    / "comparacion_configuraciones_random_forest.csv",
    index=False
)

print(
    "Comparación de configuraciones guardada correctamente."
)

informe_rf_optimizado_df.to_csv(
    RUTA_RF_OPTIMIZADO
    / "informe_clasificacion_random_forest_optimizado.csv"
)

print(
    "Informe de clasificación guardado correctamente."
)


Comparación de configuraciones guardada correctamente.
Informe de clasificación guardado correctamente.


In [34]:
# Comprobación 
print("\nARCHIVOS DEL MODELO OPTIMIZADO")
print("------------------------------")

for archivo in sorted(
    RUTA_RF_OPTIMIZADO.iterdir()
):
    tamano_mb = (
        archivo.stat().st_size
        / 1024**2
    )

    print(
        f"{archivo.name}: "
        f"{tamano_mb:.2f} MB"
    )


ARCHIVOS DEL MODELO OPTIMIZADO
------------------------------
informe_clasificacion_random_forest_optimizado.csv: 0.00 MB
matriz_confusion_normalizada_random_forest_optimizado.png: 0.12 MB
matriz_confusion_random_forest_optimizado.png: 0.11 MB
metricas_random_forest_optimizado.json: 0.00 MB
modelo_random_forest_optimizado.joblib: 71.36 MB
predicciones_random_forest_optimizado.npy: 0.57 MB
resultados_predicciones_random_forest_optimizado.parquet: 38.02 MB
